# 00 — Sanity Check (VRAM + Speed)

Run this **before** training to find out:
1. How much VRAM the model uses
2. How long one batch takes (forward + backward)
3. What `batch_size` is safe with 20% headroom
4. Estimated total training time

**No real dataset needed** — uses fake 384×384 frames so this runs in ~5 minutes.


In [ ]:
# ── Cell 1: GPU info ──────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Enable GPU runtime first.')

n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name}')
    print(f'  Total VRAM : {props.total_memory / 1e9:.1f} GB')
    print(f'  CUDA caps  : {props.major}.{props.minor}')

TOTAL_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\nUsing GPU 0 for sanity check. Total VRAM: {TOTAL_VRAM_GB:.1f} GB')


In [ ]:
# ── Cell 2: Install + clone repo ──────────────────────────────────────────────
import os, sys, shutil

REPO_DIR = '/kaggle/working/groundzero'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone https://github.com/shaunmarv3/groundzero-backend.git {REPO_DIR}

# Don't use requirements.txt — pinned versions conflict on Kaggle.
# Kaggle already has torch/numpy/PIL. Only install what's missing:
!pip install -q "transformers>=4.49.0" "peft>=0.18.0" einops wandb

sys.path.insert(0, REPO_DIR)

import importlib.util
spec = importlib.util.find_spec('app')
print(f'app module: {"FOUND ✓" if spec else "NOT FOUND"}')


In [ ]:
# ── Cell 3: HuggingFace login (needed to download SigLIP 2) ──────────────────
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except ImportError:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

from huggingface_hub import login
login(token=HF_TOKEN)
print('Logged in')


In [ ]:
# ── Cell 4: Load model + measure base VRAM ────────────────────────────────────
import torch
import torch.nn.functional as F

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

before_mb = torch.cuda.memory_allocated() / 1e6

from app.pipeline.groundzero_model import GroundZeroModel
model = GroundZeroModel(device='cuda')
model.eval()

after_mb = torch.cuda.memory_allocated() / 1e6
peak_mb  = torch.cuda.max_memory_allocated() / 1e6

TOTAL_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f'Model footprint : {after_mb - before_mb:.0f} MB')
print(f'Peak during load: {peak_mb:.0f} MB')
print(f'VRAM remaining  : {TOTAL_VRAM_GB*1000 - after_mb:.0f} MB  ({(TOTAL_VRAM_GB*1000 - after_mb)/1000:.1f} GB)')

counts = model.count_params()
print(f"Trainable: {counts['total_trainable']/1e6:.1f} M  Frozen: {counts['total_frozen']/1e6:.1f} M")


In [ ]:
# ── Cell 5: Build fake batch ───────────────────────────────────────────────────
# Creates real PIL images of the correct size — no dataset download needed.
from PIL import Image
import random

def make_fake_batch(batch_size, n_frames=150, img_size=384):
    """Simulates one batch of batch_size videos, each with n_frames frames."""
    batch = {
        'frames':       [],
        'timestamps':   [],
        'query':        [f'a person does something activity {i}' for i in range(batch_size)],
        'gt_start_idx': torch.randint(0, n_frames//2, (batch_size,)),
        'gt_end_idx':   torch.randint(n_frames//2, n_frames, (batch_size,)),
        'is_negative':  torch.zeros(batch_size, dtype=torch.bool),
        'duration':     torch.full((batch_size,), 150.0),
    }
    for _ in range(batch_size):
        # random color images — same size as real QVHighlights frames
        frames = [Image.new('RGB', (img_size, img_size),
                             color=(random.randint(0,255), random.randint(0,255), random.randint(0,255)))
                  for _ in range(n_frames)]
        batch['frames'].append(frames)
        batch['timestamps'].append(list(range(n_frames)))  # 0,1,2,...,149
    return batch

print('Fake batch builder ready')
print(f'Each batch item = {n_frames if (n_frames := 150) else 150} frames × 384×384 RGB PIL images')


In [ ]:
# ── Cell 6: VRAM scan across batch sizes ──────────────────────────────────────
import torch.nn.functional as F
from app.training.losses import combined_loss

DEVICE = 'cuda'
SAFE_HEADROOM = 0.80  # use at most 80% of VRAM

results = []

for bs in [1, 2, 4, 8, 16]:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        batch = make_fake_batch(bs)

        # ── forward pass under autocast ───────────────────────────────────────
        with torch.cuda.amp.autocast():
            emb_list, ts_list = [], []
            for i in range(bs):
                emb_i  = model.visual_encoder.encode_frames(batch['frames'][i])
                frac_i = torch.tensor([t/150.0 for t in batch['timestamps'][i]],
                                       dtype=torch.float32, device=DEVICE)
                emb_list.append(emb_i)
                ts_list.append(frac_i)

            N = max(e.shape[0] for e in emb_list)
            D = 1152
            frame_embs = torch.zeros(bs, N, D, device=DEVICE)
            timestamps = torch.ones(bs, N, device=DEVICE)
            for i, (emb, ts) in enumerate(zip(emb_list, ts_list)):
                frame_embs[i, :emb.shape[0]] = emb
                timestamps[i, :ts.shape[0]]  = ts

            query_emb = model.text_encoder.encode_queries(batch['query']).unsqueeze(1)
            start_l, end_l, conf = model(frame_embs, timestamps, query_emb)

            gt_s = batch['gt_start_idx'].to(DEVICE)
            gt_e = batch['gt_end_idx'].to(DEVICE)
            total, *_ = combined_loss(start_l, end_l, gt_s, gt_e, query_emb, frame_embs)
            conf_l = F.binary_cross_entropy(conf, torch.ones_like(conf))
            loss   = total + conf_l

        # ── backward ──────────────────────────────────────────────────────────
        scaler_tmp = torch.cuda.amp.GradScaler()
        scaler_tmp.scale(loss).backward()

        peak_gb   = torch.cuda.max_memory_allocated() / 1e9
        used_frac = peak_gb / TOTAL_VRAM_GB
        safe      = used_frac <= SAFE_HEADROOM
        results.append((bs, peak_gb, used_frac, safe, None))

        # clear gradients for next iteration
        for p in model.parameters():
            p.grad = None

    except torch.cuda.OutOfMemoryError:
        results.append((bs, None, None, False, 'OOM'))

    torch.cuda.empty_cache()

print(f'\n{"batch_size":>12}  {"peak VRAM":>12}  {"% of total":>12}  {"safe (≤80%)?":>14}')
print('-' * 56)
for bs, peak, frac, safe, err in results:
    if err:
        print(f'{bs:>12}  {"OOM":>12}  {"—":>12}  {"NO":>14}')
    else:
        mark = 'YES ✓' if safe else 'NO  ✗'
        print(f'{bs:>12}  {peak:.2f} GB{"":>5}  {frac*100:.1f}%{"":>8}  {mark:>14}')

safe_results = [(bs, peak) for bs, peak, frac, safe, err in results if safe]
if safe_results:
    recommended_bs = safe_results[-1][0]
    print(f'\nRecommended batch_size = {recommended_bs}  (largest that fits within 80% VRAM)')
else:
    print('\nEven batch_size=1 exceeds 80% VRAM — consider gradient checkpointing.')


In [ ]:
# ── Cell 7: Time one batch + extrapolate training duration ───────────────────
import time

BS_TO_TIME = recommended_bs if safe_results else 1

torch.cuda.empty_cache()
batch = make_fake_batch(BS_TO_TIME)

optimizer_tmp = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-4
)
scaler_tmp = torch.cuda.amp.GradScaler()

# Warmup pass (first pass is slower due to kernel compilation)
with torch.cuda.amp.autocast():
    emb_list = [model.visual_encoder.encode_frames(batch['frames'][i]) for i in range(BS_TO_TIME)]
_ = emb_list[0].sum()  # force sync

# Timed pass
torch.cuda.synchronize()
t0 = time.time()

with torch.cuda.amp.autocast():
    emb_list, ts_list = [], []
    for i in range(BS_TO_TIME):
        emb_i  = model.visual_encoder.encode_frames(batch['frames'][i])
        frac_i = torch.tensor([t/150.0 for t in batch['timestamps'][i]],
                               dtype=torch.float32, device=DEVICE)
        emb_list.append(emb_i)
        ts_list.append(frac_i)

    N = max(e.shape[0] for e in emb_list)
    frame_embs = torch.zeros(BS_TO_TIME, N, 1152, device=DEVICE)
    timestamps = torch.ones(BS_TO_TIME, N, device=DEVICE)
    for i, (emb, ts) in enumerate(zip(emb_list, ts_list)):
        frame_embs[i, :emb.shape[0]] = emb
        timestamps[i, :ts.shape[0]]  = ts
    query_emb = model.text_encoder.encode_queries(batch['query']).unsqueeze(1)
    start_l, end_l, conf = model(frame_embs, timestamps, query_emb)
    gt_s = batch['gt_start_idx'].to(DEVICE)
    gt_e = batch['gt_end_idx'].to(DEVICE)
    total, *_ = combined_loss(start_l, end_l, gt_s, gt_e, query_emb, frame_embs)
    loss = total + F.binary_cross_entropy(conf, torch.ones_like(conf))

optimizer_tmp.zero_grad()
scaler_tmp.scale(loss).backward()
scaler_tmp.unscale_(optimizer_tmp)
scaler_tmp.step(optimizer_tmp)
scaler_tmp.update()

torch.cuda.synchronize()
batch_time_sec = time.time() - t0

# Extrapolate
N_TRAIN       = 7445
batches_epoch = N_TRAIN / BS_TO_TIME
N_EPOCHS      = 20

time_per_epoch_h  = batches_epoch * batch_time_sec / 3600
time_total_h      = time_per_epoch_h * N_EPOCHS

print(f'batch_size        : {BS_TO_TIME}')
print(f'time per batch    : {batch_time_sec:.1f} sec')
print(f'batches per epoch : {batches_epoch:.0f}')
print(f'time per epoch    : {time_per_epoch_h:.1f} hours')
print(f'total (20 epochs) : {time_total_h:.1f} hours')
print(f'\nKaggle T4 (30h/week free) → ~{time_total_h/30:.1f} weeks')
print(f'Colab Pro A100    (faster) → likely {time_total_h/5:.1f}–{time_total_h/3:.1f} hours total (estimate)')


In [ ]:
# ── Cell 8: Summary ────────────────────────────────────────────────────────────
print('=' * 50)
print('SANITY CHECK SUMMARY')
print('=' * 50)
print(f'GPU                : {torch.cuda.get_device_name(0)}')
print(f'Total VRAM         : {TOTAL_VRAM_GB:.1f} GB')
print(f'Model footprint    : {after_mb:.0f} MB')
print(f'Recommended bs     : {recommended_bs if safe_results else 1}')
print(f'Sec per batch (bs={BS_TO_TIME}): {batch_time_sec:.1f}s')
print(f'Est. total training: {time_total_h:.0f} hours')
print('='*50)
print(f'\nSet CFG["batch_size"] = {recommended_bs if safe_results else 1} in 03_train_groundzero.ipynb')
